In [30]:
import gc
from urllib.parse import unquote

import numpy as np
import pandas as pd
from bokeh.models import ColumnDataSource, FixedTicker, HoverTool, NumeralTickFormatter, Range1d
from bokeh.plotting import figure, output_notebook, show

output_notebook()

Loading BokehJS ...

In [31]:
# ── Chromosome sizes from contigs.npy ────────────────────────────────────
contigs = np.load('../inputs/Pf9-53973-samples-100bp-npy/contigs.npy', allow_pickle=True)

chrom_sizes = {}
for row in contigs:
    c, end = row['chrom'], int(row['end'])
    if c not in chrom_sizes or end > chrom_sizes[c]:
        chrom_sizes[c] = end

del contigs
gc.collect()

CHROMS = [f'Pf3D7_{i:02d}_v3' for i in range(1, 15)] + ['Pf3D7_API_v3', 'Pf3D7_MIT_v3']
CHROMS = [c for c in CHROMS if c in chrom_sizes]
# y=0 → chr01 at top after axis inversion
CHROM_Y = {c: i for i, c in enumerate(CHROMS)}

print('Chromosomes loaded:')
for c in CHROMS:
    print(f'  {c}: {chrom_sizes[c]:,} bp')

Chromosomes loaded:
  Pf3D7_01_v3: 640,876 bp
  Pf3D7_02_v3: 947,151 bp
  Pf3D7_03_v3: 1,067,986 bp
  Pf3D7_04_v3: 1,200,495 bp
  Pf3D7_05_v3: 1,343,579 bp
  Pf3D7_06_v3: 1,418,271 bp
  Pf3D7_07_v3: 1,445,254 bp
  Pf3D7_08_v3: 1,472,853 bp
  Pf3D7_09_v3: 1,541,768 bp
  Pf3D7_10_v3: 1,687,678 bp
  Pf3D7_11_v3: 2,038,370 bp
  Pf3D7_12_v3: 2,271,497 bp
  Pf3D7_13_v3: 2,925,268 bp
  Pf3D7_14_v3: 3,291,968 bp
  Pf3D7_API_v3: 34,275 bp
  Pf3D7_MIT_v3: 5,984 bp


In [32]:
# ── Core genome regions ───────────────────────────────────────────────────
# BED columns: index  chrom  start  end  label
core_df = pd.read_csv(
    '../../assets/core-genome.bed',
    sep='\t', header=None,
    names=['chrom', 'start', 'end', 'label'],
)
core_df = core_df[core_df['chrom'].isin(CHROMS)].reset_index(drop=True)
print(f'Core regions: {len(core_df)} intervals across {core_df["chrom"].nunique()} chromosomes')
core_df.head()

Core regions: 34 intervals across 14 chromosomes


,chrom,start,end,label
0,Pf3D7_01_v3,92901,457931,Core
1,Pf3D7_01_v3,460312,575900,Core
2,Pf3D7_02_v3,105801,447300,Core
3,Pf3D7_02_v3,450451,862500,Core
4,Pf3D7_03_v3,70631,597816,Core


In [33]:
# ── Gene annotations from GFF ─────────────────────────────────────────────
# Keep gene-level features only; transcript/exon/CDS are children and would duplicate
GENE_FEATURES = {'protein_coding_gene', 'pseudogene', 'ncRNA_gene'}

rows = []
with open('../../assets/PlasmoDB-54_Pfalciparum3D7.gff') as fh:
    for line in fh:
        if line.startswith('#'):
            continue
        parts = line.rstrip('\n').split('\t')
        if len(parts) < 9:
            continue
        chrom, _, feature, start, end, _, strand, _, attrs = parts
        if feature not in GENE_FEATURES:
            continue
        attr_dict = {}
        for a in attrs.split(';'):
            if '=' in a:
                k, v = a.split('=', 1)
                attr_dict[k] = unquote(v)
        rows.append({
            'chrom':   chrom,
            'start':   int(start) - 1,  # GFF is 1-based → convert to 0-based
            'end':     int(end),
            'strand':  strand,
            'feature': feature,
            'name':    attr_dict.get('Name', attr_dict.get('ID', '')),
            'desc':    attr_dict.get('description', ''),
        })

genes_df = pd.DataFrame(rows)
genes_df = genes_df[genes_df['chrom'].isin(CHROMS)].reset_index(drop=True)
print(f'Genes loaded: {len(genes_df)}')
print(genes_df['feature'].value_counts())

Genes loaded: 5720
feature
protein_coding_gene    5318
ncRNA_gene              244
pseudogene              158
Name: count, dtype: int64


In [34]:
# ── 200 bp core-genome bins classified by distance to nearest gene ─────────
BIN_SIZE    = 200   # bp
DIST_THRESH = 1   # bp — farther than this from any gene → gene desert

bin_chunks = []
stat_rows  = []

for chrom in CHROMS:
    cgenes = genes_df[genes_df['chrom'] == chrom]
    ccore  = core_df[core_df['chrom'] == chrom]
    if ccore.empty:
        continue

    # Build bin starts for every core region on this chrom.
    # Track each bin's region end so we can clamp bin_ends correctly —
    # without this, the last bin of each region overshoots into the gap or
    # the next region, causing visual overlaps.
    starts_list      = []
    region_ends_list = []
    for _, reg in ccore.iterrows():
        s = np.arange(int(reg['start']), int(reg['end']), BIN_SIZE)
        starts_list.append(s)
        region_ends_list.append(np.full(len(s), int(reg['end'])))
    bin_starts  = np.concatenate(starts_list)
    region_ends = np.concatenate(region_ends_list)
    bin_ends    = np.minimum(bin_starts + BIN_SIZE, region_ends)

    # Vectorised nearest-gene distance using bin extents (not midpoints).
    # For bin [bs, be] and gene [gs, ge]:
    #   dist = gs - be  if bin is entirely left of gene
    #   dist = bs - ge  if bin is entirely right of gene
    #   dist = 0        if they overlap
    # Combined: max(0, max(gs - be, bs - ge))
    # Using midpoints would mis-classify bins that partially overlap a gene.
    if len(cgenes) == 0:
        min_dists = np.full(len(bin_starts), np.inf)
    else:
        gs = cgenes['start'].values.astype(float)[np.newaxis, :]  # (1, G)
        ge = cgenes['end'].values.astype(float)[np.newaxis, :]
        bs = bin_starts[:, np.newaxis]                             # (M, 1)
        be = bin_ends[:, np.newaxis]                               # (M, 1)
        min_dists = np.maximum(0.0, np.maximum(gs - be, bs - ge)).min(axis=1)
        del gs, ge, bs, be
        gc.collect()

    is_red  = min_dists > DIST_THRESH
    n_red   = int(is_red.sum())
    n_norm  = int((~is_red).sum())
    n_total = len(is_red)

    stat_rows.append(dict(
        chrom       = chrom,
        total       = n_total,
        near_gene   = n_norm,
        gene_desert = n_red,
        pct_desert  = 100.0 * n_red / n_total if n_total else 0.0,
    ))

    # Only store gene-desert (red) bins for the plot layer;
    # near-gene bins are visually covered by the existing core overlay.
    red_mask = is_red
    bin_chunks.append(pd.DataFrame(dict(
        chrom    = chrom,
        start    = bin_starts[red_mask],
        end      = bin_ends[red_mask],
        min_dist = min_dists[red_mask].astype(int),
    )))
    del bin_starts, bin_ends, region_ends, min_dists, is_red, red_mask
    gc.collect()

desert_bins_df = pd.concat(bin_chunks, ignore_index=True) if bin_chunks else pd.DataFrame()
stats_df       = pd.DataFrame(stat_rows)

del bin_chunks
gc.collect()

print(f'{"Chrom":<22} {"Total":>8} {"Near gene":>10} {"Desert":>8} {"% Desert":>9}')
print('-' * 62)
for _, r in stats_df.iterrows():
    print(f'{r["chrom"]:<22} {r["total"]:>8,} {r["near_gene"]:>10,} {r["gene_desert"]:>8,} {r["pct_desert"]:>8.1f}%')

Chrom                     Total  Near gene   Desert  % Desert
--------------------------------------------------------------
Pf3D7_01_v3               2,404      2,003      401     16.7%
Pf3D7_02_v3               3,769      3,325      444     11.8%
Pf3D7_03_v3               4,650      4,106      544     11.7%
Pf3D7_04_v3               4,666      3,933      733     15.7%
Pf3D7_05_v3               6,411      5,596      815     12.7%
Pf3D7_06_v3               6,004      5,350      654     10.9%
Pf3D7_07_v3               6,025      5,331      694     11.5%
Pf3D7_08_v3               6,251      5,430      821     13.1%
Pf3D7_09_v3               6,962      6,062      900     12.9%
Pf3D7_10_v3               7,515      6,640      875     11.6%
Pf3D7_11_v3               9,456      8,247    1,209     12.8%
Pf3D7_12_v3              10,153      8,922    1,231     12.1%
Pf3D7_13_v3              13,577     12,036    1,541     11.4%
Pf3D7_14_v3              16,083     14,081    2,002     12.4%


In [35]:
stats_df.near_gene.sum()

np.int64(91062)

In [36]:
# ── Build Gantt-style genome overview ────────────────────────────────────
N = len(CHROMS)
MAX_LEN = max(chrom_sizes.values())

BACKBONE_H = 0.85
CORE_H     = 0.85
GENE_H     = 0.18

GENE_TRACKS = [
    ('protein_coding_gene', '+', +0.30, '#E07B2A', 'Protein-coding (+ strand)'),
    ('protein_coding_gene', '-', +0.10, '#3B9BBB', 'Protein-coding (− strand)'),
    ('pseudogene',          None, -0.10, '#9B59B6', 'Pseudogene'),
    ('ncRNA_gene',          None, -0.30, '#27AE60', 'ncRNA gene'),
]

p = figure(
    width=1400,
    height=max(500, N * 50 + 100),
    x_range=Range1d(0, MAX_LEN),
    y_range=Range1d(N - 0.5, -0.5),
    title='P. falciparum 3D7 — genome overview',
    tools='pan,box_zoom,wheel_zoom,reset,save',
    toolbar_location='above',
)
p.xaxis.axis_label = 'Genomic position (bp)'
p.yaxis.ticker = FixedTicker(ticks=list(range(N)))
p.yaxis.major_label_overrides = {i: c for i, c in enumerate(CHROMS)}
p.xaxis.formatter = NumeralTickFormatter(format='0,0')
p.grid.grid_line_color = None
p.background_fill_color = '#F5F5F5'

# ── Layer 1: chromosome backbone ──────────────────────────────────────────
backbone_src = ColumnDataSource(dict(
    y=[CHROM_Y[c] for c in CHROMS],
    left=[0] * N,
    right=[chrom_sizes[c] for c in CHROMS],
))
p.hbar(
    y='y', left='left', right='right', height=BACKBONE_H,
    source=backbone_src,
    color='#D8D8D8', line_color='#AAAAAA', line_width=0.5,
    legend_label='Subtelomeric / non-core',
)
# No hover tool for chromosome backbone — it's distracting

# ── Layer 2: core genome regions ──────────────────────────────────────────
core_src = ColumnDataSource(dict(
    y=[CHROM_Y[c] for c in core_df['chrom']],
    left=core_df['start'].tolist(),
    right=core_df['end'].tolist(),
    chrom=core_df['chrom'].tolist(),
    start_fmt=[f'{v:,}' for v in core_df['start']],
    end_fmt=[f'{v:,}' for v in core_df['end']],
))
core_rend = p.hbar(
    y='y', left='left', right='right', height=CORE_H,
    source=core_src,
    color='#BDD7EE', line_color=None,
    legend_label='Core genome (near gene)',
)
p.add_tools(HoverTool(renderers=[core_rend], tooltips=[
    ('Region', 'Core genome'),
    ('Chrom', '@chrom'),
    ('Start', '@start_fmt'), ('End', '@end_fmt'),
]))

# ── Layer 3: gene-desert bins (>400 bp from any gene) ────────────────────
if not desert_bins_df.empty:
    desert_src = ColumnDataSource(dict(
        y        = [CHROM_Y[c] for c in desert_bins_df['chrom']],
        left     = desert_bins_df['start'].tolist(),
        right    = desert_bins_df['end'].tolist(),
        chrom    = desert_bins_df['chrom'].tolist(),
        start_fmt= [f'{v:,}' for v in desert_bins_df['start']],
        end_fmt  = [f'{v:,}' for v in desert_bins_df['end']],
        min_dist = desert_bins_df['min_dist'].tolist(),
    ))
    desert_rend = p.hbar(
        y='y', left='left', right='right', height=CORE_H,
        source=desert_src,
        color='#E74C3C', line_color=None, alpha=0.80,
        legend_label='Core genome (gene desert, >400 bp from gene)',
    )
    p.add_tools(HoverTool(renderers=[desert_rend], tooltips=[
        ('Type',         'Gene desert bin (200 bp)'),
        ('Chrom',        '@chrom'),
        ('Start',        '@start_fmt'),
        ('End',          '@end_fmt'),
        ('Nearest gene', '@min_dist{0,0} bp away'),
    ]))

# ── Layers 4+: gene tracks ────────────────────────────────────────────────
for feature, strand, offset, color, legend_label in GENE_TRACKS:
    mask = genes_df['feature'] == feature
    if strand is not None:
        mask = mask & (genes_df['strand'] == strand)
    sdf = genes_df[mask]
    if sdf.empty:
        continue

    src = ColumnDataSource(dict(
        y=[CHROM_Y[c] + offset for c in sdf['chrom']],
        left=sdf['start'].tolist(),
        right=sdf['end'].tolist(),
        chrom=sdf['chrom'].tolist(),
        name=sdf['name'].tolist(),
        desc=sdf['desc'].tolist(),
        strand=sdf['strand'].tolist(),
        feature=sdf['feature'].tolist(),
        start_fmt=[f'{v:,}' for v in sdf['start']],
        end_fmt=[f'{v:,}' for v in sdf['end']],
    ))
    rend = p.hbar(
        y='y', left='left', right='right', height=GENE_H,
        source=src,
        color=color, line_color=None, alpha=0.85,
        legend_label=legend_label,
    )
    p.add_tools(HoverTool(renderers=[rend], tooltips=[
        ('Name',        '@name'),
        ('Type',        '@feature'),
        ('Chrom',       '@chrom'),
        ('Start',       '@start_fmt'),
        ('End',         '@end_fmt'),
        ('Strand',      '@strand'),
        ('Description', '@desc'),
    ]))

p.legend.location = 'bottom_right'
p.legend.click_policy = 'hide'
p.legend.background_fill_alpha = 0.9

show(p)